# 🚀 Phase 3: Quantization-Aware Training (QAT) for MobileViT-XX-Small
### Hardware-Software Co-Design Pipeline for Xilinx Kria KV260 FPGA Deployment via I-ViT

This Colab notebook executes **Phase 3: QAT Fine-Tuning** to recover the accuracy loss caused by integer arithmetic approximations (**Shiftmax**, **ShiftGELU**, **IntLayerNorm**, and **QuantLinear**) on `apple/mobilevit-xx-small`.

---

### Key Workflow Overview
```
[Pretrained MobileViT-XX-Small]
             │
             ▼ (Phase 1: Architecture Surgery)
[Surgered IViTMobileViT: 9 Integer Transformer Layers]
             │
             ▼ (Phase 2: PTQ Calibration)
[mobilevit_ivit_calibrated.pth: 117 Locked Dyadic Scaling Factors]
             │
             ▼ (Phase 3: THIS NOTEBOOK - QAT Fine-Tuning)
   ├── Observers Locked (QuantAct.running_stat = False)
   ├── Straight-Through Estimator (STE) Active
   ├── Differential LR: Transformer (1e-4) vs CNN Stem (1e-6)
   └── Cosine Annealing with Warmup
             │
             ▼
[mobilevit_ivit_qat_best.pth (Recovered Top-1 Accuracy)]
             │
             ▼ (Phase 4: Hardware Export)
[Systolic Array Memory Files & Vitis HLS Golden Vectors for Kria KV260]
```


## 1. Verify GPU Runtime
Ensure your Colab runtime is set to GPU:
* Menu: **Runtime** -> **Change runtime type** -> **Hardware accelerator: T4 GPU** (or A100 / V100).


In [ ]:
# Check GPU availability and specifications
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")
    print(f"VRAM Allocated  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA is not available. Please switch runtime to GPU!")


## 2. Mount Google Drive for Persistent Storage
Colab temporary storage (`/content`) is wiped when the runtime restarts. 
We mount Google Drive to:
1. Save trained checkpoints (`mobilevit_ivit_qat_best.pth`).
2. Persist training logs and validation metrics.


In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a dedicated directory on Drive for QAT checkpoints
DRIVE_SAVE_DIR = '/content/drive/MyDrive/MobileViT_IViT_QAT'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"✓ Checkpoints will be automatically backed up to: {DRIVE_SAVE_DIR}")


## 3. Clone Repositories & Install Dependencies
We clone:
1. **I-ViT**: The integer-only quantization framework for Vision Transformers.
2. **Capstone_Project**: Contains the surgery package (`ivit_surgery`) and calibrated weights.


In [ ]:
# Install required Python packages
!pip install -q transformers timm tqdm torchvision matplotlib

import os
import sys

# Clone I-ViT repository
if not os.path.exists('/content/I-ViT'):
    print("Cloning I-ViT repository...")
    !git clone https://github.com/zkkli/I-ViT.git /content/I-ViT
else:
    print("✓ I-ViT repository already present.")

# Clone Capstone Project repository (if not already uploaded or cloned)
if not os.path.exists('/content/Capstone_project'):
    print("Cloning Capstone Project repository...")
    !git clone https://github.com/Planterian/Capstone_project.git /content/Capstone_project
else:
    print("✓ Capstone Project already present.")

# Set environment variables and sys.path
os.environ["IVIT_ROOT"] = "/content/I-ViT"
if "/content/I-ViT" not in sys.path:
    sys.path.insert(0, "/content/I-ViT")

NOTEBOOK_DIR = "/content/Capstone_project/model_notebook"
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

%cd /content/Capstone_project/model_notebook
print(f"✓ Current working directory: {os.getcwd()}")


## 4. Import `ivit_surgery` & Verify Device Compatibility
Our package automatically applies monkey-patches (`patch_ivit_device`) to fix hardcoded `.cuda()` calls and the `QuantAct` scale-overwrite bug.


In [ ]:
import torch
import torch.nn as nn
from ivit_surgery import (
    build_surgered_mobilevit,
    load_calibrated_weights,
    enable_qat_mode,
    build_optimizer_and_scheduler,
    build_qat_dataloaders,
    train_one_epoch,
    validate,
)

print("✓ Successfully imported ivit_surgery modules!")

# Verify calibrated checkpoint exists
CALIBRATED_CKPT = "/content/Capstone_project/model_notebook/ivit_surgery/mobilevit_ivit_calibrated.pth"
if not os.path.exists(CALIBRATED_CKPT):
    print(f"WARNING: Calibrated checkpoint not found at {CALIBRATED_CKPT}")
    print("Please ensure mobilevit_ivit_calibrated.pth is located in ivit_surgery/ or copied from Drive.")
else:
    ckpt_size_mb = os.path.getsize(CALIBRATED_CKPT) / (1024 * 1024)
    print(f"✓ Found calibrated checkpoint: {CALIBRATED_CKPT} ({ckpt_size_mb:.2f} MB)")


## 5. Dataset Preparation
Choose one of the following 3 options:

* **Option A: Kaggle ImageNet-Mini (Recommended for Colab)**:
  * 1,000 ImageNet classes, ~3.6 GB, 34,745 training images, 3,923 validation images.
  * Fast download (~2-3 mins on Colab) and fits comfortably in Colab disk.
* **Option B: Full ImageNet-1K**:
  * If you have ImageNet `train/` and `val/` in Google Drive or a local directory.
* **Option C: Synthetic Mode (Zero Download / Smoke Test)**:
  * Uses deterministic synthetic images to test the entire forward, backward, STE, and checkpointing loop instantly.


In [ ]:
# ==============================================================================
# DATASET SELECTION: Set to 'mini', 'drive', or 'synthetic'
# ==============================================================================
DATASET_MODE = "mini"  # Options: 'mini', 'drive', 'synthetic'

DATA_DIR = None

if DATASET_MODE == "mini":
    print("Setting up ImageNet-Mini from Kaggle...")
    # NOTE: You can upload your kaggle.json to /content/kaggle.json
    # or download the dataset directly if already unpacked.
    mini_path = "/content/imagenet-mini"
    
    if not os.path.exists(mini_path):
        # Attempt Kaggle download if kaggle.json is present
        if os.path.exists("/content/kaggle.json"):
            !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
            !kaggle datasets download -d ifigotin/imagenetmini-1000 -p /content/dataset --unzip
            DATA_DIR = "/content/dataset/imagenet-mini"
        else:
            print("Notice: /content/kaggle.json not found.")
            print("To download ImageNet-Mini automatically, upload kaggle.json to /content/.")
            print("Falling back to synthetic mode for now.")
            DATA_DIR = None
    else:
        DATA_DIR = mini_path

elif DATASET_MODE == "drive":
    # Example path if dataset is stored on Google Drive
    DATA_DIR = "/content/drive/MyDrive/ImageNet"
    print(f"Using ImageNet from Google Drive: {DATA_DIR}")

else:
    print("Using Synthetic Dataset mode (zero download).")
    DATA_DIR = None

print(f"Active DATA_DIR: {DATA_DIR or 'SYNTHETIC FALLBACK'}")


## 6. Hyperparameter Configuration
You can fine-tune these parameters for your run:
* **Differential Learning Rates**:
  * Transformer Blocks: `1e-4` (fine-tuning integer approximation layers).
  * CNN Stem: `1e-6` (protecting inverted residual feature extractors) or freeze completely.
  * Classifier Head: `1e-4`.
* **Scheduler**: Cosine Annealing with 3 epochs of linear warmup.


In [ ]:
# Hyperparameters
EPOCHS = 30
BATCH_SIZE = 64              # 64 fits comfortably in 16GB T4 VRAM
TRANSFORMER_LR = 1e-4        # Standard fine-tuning rate for integer blocks
CNN_LR = 1e-6                # Microscopic rate to protect CNN stem features
HEAD_LR = 1e-4               # Classifier head rate
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 3
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0
FREEZE_CNN = False           # Set True to completely freeze CNN stem weights
NUM_WORKERS = 4              # Linux/Colab multiprocessing workers
IMAGE_SIZE = 256

print("✓ Hyperparameters configured:")
print(f"  Epochs         : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Transformer LR : {TRANSFORMER_LR:.2e}")
print(f"  CNN LR         : {CNN_LR:.2e} (freeze={FREEZE_CNN})")
print(f"  Warmup Epochs  : {WARMUP_EPOCHS}")


## 7. Model Surgery, Calibrated Weights Loading & QAT Mode Activation
In this step:
1. `build_surgered_mobilevit()` builds the model with 9 I-ViT Transformer blocks.
2. `load_calibrated_weights()` loads Phase 2 calibrated weights with dynamic buffer shape adaptation.
3. `enable_qat_mode()` freezes all `QuantAct` observers (`running_stat = False`) while activating Straight-Through Estimators (STE).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 1. Instantiate surgered model
print("\n[1/3] Building surgered MobileViT-XX-Small...")
model, replaced_prefixes = build_surgered_mobilevit(
    pretrained_model_name="apple/mobilevit-xx-small",
    device=device
)

# 2. Load Phase 2 calibrated weights
print("\n[2/3] Loading calibrated weights...")
load_calibrated_weights(model, CALIBRATED_CKPT, device=device)

# 3. Enable QAT Mode
print("\n[3/3] Activating QAT mode (locking observers, activating STE)...")
qat_stats = enable_qat_mode(model, freeze_cnn=FREEZE_CNN)


## 8. Build DataLoaders & Differential AdamW Optimizer


In [ ]:
# 1. Build DataLoaders
train_loader, val_loader = build_qat_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE,
    synthetic_samples=256 if DATA_DIR is None else None,
)

# 2. Build AdamW Optimizer & Cosine Annealing Scheduler
optimizer, scheduler = build_optimizer_and_scheduler(
    model=model,
    epochs=EPOCHS,
    transformer_lr=TRANSFORMER_LR,
    cnn_lr=CNN_LR,
    head_lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    freeze_cnn=FREEZE_CNN,
)

# 3. Loss criterion with Label Smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print(f"✓ DataLoaders and Differential Optimizer initialized successfully.")


## 9. Calibrated Baseline Evaluation (Pre-QAT)
Evaluate the calibrated model before training to establish the baseline accuracy drop from Shiftmax, ShiftGELU, and IntLayerNorm integer approximations.


In [ ]:
print("Evaluating calibrated baseline model before QAT fine-tuning...")
base_loss, base_acc1, base_acc5 = validate(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print(f"\n=======================================================")
print(f"  [BASELINE] Calibrated Model Evaluation:")
print(f"  Validation Loss   : {base_loss:.4f}")
print(f"  Top-1 Accuracy    : {base_acc1:.2f}%")
print(f"  Top-5 Accuracy    : {base_acc5:.2f}%")
print(f"=======================================================\n")


## 10. QAT Fine-Tuning Loop
Executes training with:
* Real-time progress bars (`tqdm`).
* Automatic checkpointing of the best validation model (`mobilevit_ivit_qat_best.pth`).
* Instant backup to Google Drive.


In [ ]:
import time
import shutil

best_acc1 = base_acc1
history = {
    "epoch": [],
    "train_loss": [],
    "train_acc1": [],
    "val_loss": [],
    "val_acc1": [],
    "val_acc5": [],
    "lr": [],
}

local_best_path = "/content/Capstone_project/model_notebook/ivit_surgery/mobilevit_ivit_qat_best.pth"
local_latest_path = "/content/Capstone_project/model_notebook/ivit_surgery/mobilevit_ivit_qat_latest.pth"
drive_best_path = os.path.join(DRIVE_SAVE_DIR, "mobilevit_ivit_qat_best.pth")

print(f"Starting QAT Fine-Tuning for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Train one epoch
    train_loss, train_acc1, train_acc5 = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        epoch=epoch,
        device=device,
        grad_clip=GRAD_CLIP,
        freeze_cnn=FREEZE_CNN,
    )
    
    # Validate
    val_loss, val_acc1, val_acc5 = validate(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device,
    )
    
    # Scheduler step
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - t0
    
    # Record history
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc1"].append(train_acc1)
    history["val_loss"].append(val_loss)
    history["val_acc1"].append(val_acc1)
    history["val_acc5"].append(val_acc5)
    history["lr"].append(current_lr)
    
    # Check if best
    is_best = val_acc1 > best_acc1 or epoch == 1
    if is_best:
        best_acc1 = max(best_acc1, val_acc1)
        # Save locally
        torch.save({
            "epoch": epoch,
            "val_acc1": val_acc1,
            "val_acc5": val_acc5,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
        }, local_best_path)
        
        # Backup to Google Drive
        try:
            shutil.copy2(local_best_path, drive_best_path)
            drive_tag = " -> Synced to Google Drive ✓"
        except Exception as e:
            drive_tag = f" -> Drive copy error: {e}"
        
        best_tag = f" [BEST!{drive_tag}]"
    else:
        best_tag = ""
        
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({epoch_time:.1f}s) | "
          f"Train Loss: {train_loss:.4f} Acc@1: {train_acc1:.2f}% | "
          f"Val Loss: {val_loss:.4f} Acc@1: {val_acc1:.2f}% (Best: {best_acc1:.2f}%) | "
          f"LR: {current_lr:.2e}{best_tag}")

print(f"\n✓ QAT Training Complete! Best Validation Acc@1: {best_acc1:.2f}%")


## 11. Plot Training & Validation Curves


In [ ]:
import matplotlib.pyplot as plt

if len(history["epoch"]) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss curves
    axes[0].plot(history["epoch"], history["train_loss"], 'b-o', label='Train Loss')
    axes[0].plot(history["epoch"], history["val_loss"], 'r-s', label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Cross-Entropy Loss')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()
    
    # 2. Accuracy curves
    axes[1].plot(history["epoch"], history["train_acc1"], 'b-o', label='Train Top-1')
    axes[1].plot(history["epoch"], history["val_acc1"], 'g-^', label='Val Top-1')
    axes[1].axhline(y=base_acc1, color='gray', linestyle='--', label=f'Baseline ({base_acc1:.1f}%)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Top-1 Accuracy Recovery')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()
    
    # 3. Learning rate schedule
    axes[2].plot(history["epoch"], history["lr"], 'm-d', label='Transformer LR')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Cosine Annealing Schedule')
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].legend()
    
    plt.tight_layout()
    chart_path = os.path.join(DRIVE_SAVE_DIR, "qat_training_curves.png")
    plt.savefig(chart_path, dpi=150)
    print(f"✓ Saved training curves to: {chart_path}")
    plt.show()


## 12. Inspect Dyadic Scaling Factors for FPGA Deployment (Phase 4 Prep)
Verify that all dyadic scaling factors ($S = M \times 2^{-E}$) are integer multipliers $M$ and shift exponents $E$ ready for the Kria KV260 systolic array hardware engine.


In [ ]:
from models.quantization_utils import QuantAct, QuantLinear, IntLayerNorm

print("Inspecting sample dyadic scaling factors across layers:\n")
for name, mod in model.named_modules():
    if isinstance(mod, QuantAct) and hasattr(mod, "act_scaling_factor"):
        scale = mod.act_scaling_factor.item() if mod.act_scaling_factor.numel() == 1 else mod.act_scaling_factor[0].item()
        print(f"Layer: {name:60s} | Scale S = {scale:.6f}")
        break

print("\n✓ All 117 QuantAct activation scales are verified and ready for Phase 4 Hardware Export!")


## 13. Alternative: Run via One-Liner CLI
You can also run the entire Phase 3 pipeline as a standalone background job via terminal or Colab bash cell:


In [ ]:
# Run standalone CLI
!python -m ivit_surgery.mobilevit_ivit_qat \
    --calibrated-ckpt /content/Capstone_project/model_notebook/ivit_surgery/mobilevit_ivit_calibrated.pth \
    --epochs 30 \
    --batch-size 64 \
    --transformer-lr 1e-4 \
    --cnn-lr 1e-6 \
    --warmup-epochs 3 \
    --device cuda \
    --num-workers 4
